In [ ]:
# Notebook 5/9 - Raw Database Construction with GROBID (Test + Control)
#
# Purpose:
# - Extract structured text sections from PDF documents (retracted/test and matched control articles)
#   using a locally hosted GROBID service.
# - Build a raw, case-level database that will later be merged, cleaned, and used for modelling
#   and statistical analysis in subsequent notebooks.
#
# What this notebook does:
# 1) Installs system dependencies and Python packages required to run GROBID locally in Colab.
# 2) Starts a local GROBID service inside the Colab runtime.
# 3) Loads case metadata and builds an index of available PDFs for:
#    - Test (retracted) articles
#    - Control (matched) articles
# 4) Runs GROBID extraction to obtain structured sections (e.g., title, abstract, body, references).
# 5) Writes/updates a raw database CSV in an append-safe way, supporting resume after interruptions.
# 6) Produces audit summaries used for reporting and quality checks.
#
# Inputs (Google Drive)
# - Retracted PDFs folder: oa_pdfs_all/<code>.pdf
# - Control PDFs folder:   control_oa_pdfs_all/<code>.pdf
# - Case metadata file(s): paths configured in the Configuration section
#
# Outputs (Google Drive):
# - Raw database CSV containing extracted sections for test + control articles
# - Intermediate indexes and logs as configured
#
# Execution:
# - Run cells top-to-bottom.
# - If interrupted, rerun from the start; the append-safe output design prevents rework and duplication.
#
# This notebook is part of a larger, multi-stage data acquisition and
# analysis pipeline and is designed to be fully reproducible.

In [ ]:
# 1) Mount Google Drive + Install dependencies

from google.colab import drive
drive.mount("/content/drive")

!pip -q install pymupdf==1.24.9 rapidfuzz==3.9.6 tqdm==4.66.5 pandas==2.2.2 openpyxl==3.1.5 requests==2.32.3 trafilatura==1.12.2 beautifulsoup4==4.12.3 lxml==5.3.0

# Java for running GROBID
!apt-get -qq update
!apt-get -qq install -y default-jre unzip

In [ ]:
# 2) Imports

import os, re, time, requests, subprocess, textwrap, json, hashlib, shutil, logging, fitz, trafilatura
import pandas as pd
import numpy as np
from tqdm import tqdm
from rapidfuzz import fuzz
from bs4 import BeautifulSoup

In [ ]:
# 3) Start GROBID server

GROBID_DIR = "/content/grobid"

# Clean previous runs
!pkill -f "grobid.*jetty" >/dev/null 2>&1 || true
!pkill -f "org.eclipse.jetty" >/dev/null 2>&1 || true

# Get GROBID source (stable tag)
if not os.path.exists(GROBID_DIR):
    !git clone -q --depth 1 --branch 0.8.1 https://github.com/kermitt2/grobid.git {GROBID_DIR}

# Build
%cd /content/grobid
!./gradlew -q clean install

# Start the server in background (Jetty)
# The server will expose: http://localhost:8070
!nohup ./gradlew -q run > /content/grobid_server.log 2>&1 &

# Wait for readiness
for _ in range(120):  # allow longer for first start
    try:
        r = requests.get("http://localhost:8070/api/isalive", timeout=2)
        if r.status_code == 200:
            print("GROBID is active)")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print("GROBID did not become ready. Last logs:")
    !tail -n 120 /content/grobid_server.log || true
    raise RuntimeError("GROBID did not become ready in time.")

In [ ]:
# 4) GROBID API Helpers

GROBID_URL = "http://localhost:8070/api"

def grobid_process_fulltext(pdf_path, timeout=120):
    """
    Returns TEI XML (string) or "" if fails.
    """
    try:
        with open(pdf_path, "rb") as f:
            files = {"input": (os.path.basename(pdf_path), f, "application/pdf")}
            r = requests.post(
                f"{GROBID_URL}/processFulltextDocument",
                files=files,
                data={"consolidateHeader": "1", "consolidateCitations": "0", "teiCoordinates": "0"},
                timeout=timeout
            )
        if r.status_code != 200:
            return ""
        return r.text
    except Exception:
        return ""

def _soup_text(node):
    if not node:
        return ""
    return node.get_text("\n", strip=True)

def grobid_extract_sections_from_tei(tei_xml):
    """
    Map GROBID TEI -> title, abstract, introduction, methods, results, discussion.
    Heuristic mapping using <div><head> keywords.
    """
    out = { "title":"", "abstract":"", "introduction":"", "methods":"", "results":"", "discussion":"" }
    if not tei_xml:
        return out, "grobid:no_tei"

    soup = BeautifulSoup(tei_xml, "xml")

    # Title
    title_node = soup.find("titleStmt")
    if title_node:
        t = title_node.find("title")
        out["title"] = _soup_text(t)

    # Abstract
    abs_node = soup.find("abstract")
    out["abstract"] = _soup_text(abs_node)

    body = soup.find("body")
    if not body:
        return out, "grobid:no_body"

    buckets = {
        "introduction": ["introduction", "background"],
        "methods": ["methods", "materials and methods", "methodology", "experimental", "study design", "patients and methods", "statistical analysis"],
        "results": ["results", "findings", "analysis", "outcomes"],
        "discussion": ["discussion", "conclusion", "conclusions", "discussion and conclusions", "interpretation"],
    }

    for div in body.find_all("div"):
        head = div.find("head")
        head_txt = (_soup_text(head) or "").lower()
        if not head_txt:
            continue

        chosen = None
        for canon, keys in buckets.items():
            if any(k in head_txt for k in keys):
                chosen = canon
                break

        if chosen:
            ps = div.find_all("p")
            txt = "\n".join([_soup_text(p) for p in ps if _soup_text(p)])
            if txt:
                out[chosen] = (out[chosen] + "\n" + txt).strip()

    return out, "grobid:tei_ok"

def grobid_has_meaningful_content(secs, min_chars=1500):
    total = sum(len((secs.get(k,"") or "").strip()) for k in ["abstract","introduction","methods","results","discussion"])
    return total >= min_chars

In [ ]:
# 5) Configuration

BASE_DIR = "/content/drive/MyDrive/Dissertation"

# INPUTS
OA_TEST_DIR     = os.path.join(BASE_DIR, "oa_pdfs_all")
OA_CONTROL_DIR  = os.path.join(BASE_DIR, "control_oa_pdfs_all")
RW_XLSX         = os.path.join(BASE_DIR, "retraction_watch_database.xlsx")

CONTROL_LOG_CSV = os.path.join(BASE_DIR, "control_download_log_all_new.csv")

# OUTPUTS
RAW_DB_CSV      = os.path.join(BASE_DIR, "raw_fulltext_db.csv")
HEADING_MAP_CSV = os.path.join(BASE_DIR, "section_heading_mapping.csv")
HEADING_SCAN_CSV= os.path.join(BASE_DIR, "observed_headings_sample.csv")

# PERFORMANCE/ SCALE
MAX_PAGES_SCAN_HEADINGS = 3
MAX_PAGES_EXTRACT_TEXT  = None
PDF_CHAR_LIMIT_PER_PAPER = 2_000_000

# INTERNET FALLBACK (fast + OA only)
USE_INTERNET_FALLBACK = True
OPENALEX_MAILTO = "xxxxxxxxxxxxx"   # required by OpenAlex etiquette
UNPAYWALL_EMAIL = "xxxxxxxxxxxxx"   # required by Unpaywall

print("Config loaded")
print("Test dir:", OA_TEST_DIR)
print("Control dir:", OA_CONTROL_DIR)
print("RW:", RW_XLSX)
print("Raw DB:", RAW_DB_CSV)

In [ ]:
# 6) Load RW metadata + helpers (no pycountry needed)

rw = pd.read_excel(RW_XLSX, dtype=str).fillna("")
rw.columns = [c.strip() for c in rw.columns]

# 6.1) Schema normalization
# The file has OriginalPaperDOI (not DOI). Use OriginalPaperDOI as the paper DOI.
if "DOI" not in rw.columns:
    if "OriginalPaperDOI" in rw.columns:
        rw["DOI"] = rw["OriginalPaperDOI"]
    else:
        rw["DOI"] = ""  # keep pipeline alive even if missing entirely

# Some RW exports label Title as OriginalPaperTitle; support both
if "Title" not in rw.columns and "OriginalPaperTitle" in rw.columns:
    rw["Title"] = rw["OriginalPaperTitle"]

# Some exports may have Journal as OriginalPaperJournal; support both
if "Journal" not in rw.columns and "OriginalPaperJournal" in rw.columns:
    rw["Journal"] = rw["OriginalPaperJournal"]

# Some exports may have Publisher as OriginalPaperPublisher; support both
if "Publisher" not in rw.columns and "OriginalPaperPublisher" in rw.columns:
    rw["Publisher"] = rw["OriginalPaperPublisher"]

# 6.2) Required columns (after normalization)
REQ = ["Record ID", "Title", "DOI", "Journal", "Publisher", "Subject", "ArticleType", "OriginalPaperDate", "Country"]
missing = [c for c in REQ if c not in rw.columns]
if missing:
    raise ValueError(f"RW file missing columns: {missing}. Found: {rw.columns.tolist()}")

rw["Record ID"] = rw["Record ID"].astype(str).str.strip()

def extract_year(s):
    s = str(s or "")
    m = re.search(r"(19|20)\d{2}", s)
    return m.group(0) if m else ""

rw["year"] = rw["OriginalPaperDate"].apply(extract_year)

def norm_text(x):
    x = str(x or "").strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x

def norm_doi(x):
    x = norm_text(x)
    x = x.replace("https://doi.org/", "").replace("http://doi.org/", "")
    return x

# country to ISO2 (extend as needed)
COUNTRY_NAME_TO_ISO2 = {
    "united states":"US","usa":"US","us":"US","u.s.":"US","u.s.a.":"US",
    "united kingdom":"GB","uk":"GB","great britain":"GB","britain":"GB",
    "ireland":"IE","portugal":"PT","spain":"ES","france":"FR","germany":"DE","italy":"IT",
    "netherlands":"NL","belgium":"BE","switzerland":"CH","austria":"AT","sweden":"SE","norway":"NO",
    "denmark":"DK","finland":"FI","poland":"PL","greece":"GR","china":"CN","japan":"JP","india":"IN",
    "canada":"CA","australia":"AU","new zealand":"NZ","brazil":"BR",
}
def country_iso2(name):
    s = norm_text(name)
    return COUNTRY_NAME_TO_ISO2.get(s, "")

rw["case_country_iso2"] = rw["Country"].apply(country_iso2)

# normalized fields
for col in ["Journal","Publisher","Subject","ArticleType"]:
    rw[col+"_norm"] = rw[col].apply(norm_text)

rw["doi_norm"] = rw["DOI"].apply(norm_doi)

# lookup by code
rw_by_code = rw.set_index("Record ID")

print("RW rows:", len(rw))
print("Using DOI column from:", "OriginalPaperDOI" if "OriginalPaperDOI" in rw.columns else "DOI")

In [ ]:
# 7) Scan folders, build TEST + CONTROL index, append-only behavior

# Robust file read helpers
def safe_open_read(path, nbytes, retries=4, sleep_s=1.0):
    """
    Read up to nbytes from file, retrying on transient Drive errors.
    Returns bytes or raises last exception.
    """
    last_err = None
    for i in range(retries):
        try:
            with open(path, "rb") as f:
                return f.read(nbytes)
        except Exception as e:
            last_err = e
            time.sleep(sleep_s * (i + 1))
    raise last_err

def sha1_file_safe(path, max_bytes=2_000_000):
    """
    Stable sha1 over first max_bytes. If Drive glitches, returns "" instead of crashing.
    """
    try:
        data = safe_open_read(path, max_bytes, retries=5, sleep_s=1.0)
        h = hashlib.sha1()
        h.update(data)
        return h.hexdigest()
    except Exception as e:
        # Do NOT crash the notebook over a transient Drive mount issue
        print(f"sha1 skip (Drive read error): {path} :: {type(e).__name__}: {e}")
        return ""

def list_pdfs(folder):
    if not os.path.exists(folder):
        return []
    try:
        return sorted([f for f in os.listdir(folder) if f.lower().endswith(".pdf")])
    except Exception as e:
        # If Drive hiccups, fail softly
        print(f"list_pdfs failed for {folder}: {type(e).__name__}: {e}")
        return []

def parse_test_code(filename):
    return os.path.splitext(filename)[0].strip()

def parse_control_case_code(filename):
    base = os.path.splitext(filename)[0].strip()
    return base[1:] if base.startswith("C") else base

# Load control log if exists
control_log = None
if os.path.exists(CONTROL_LOG_CSV):
    control_log = pd.read_csv(CONTROL_LOG_CSV, dtype=str).fillna("")
    print("Loaded control log:", len(control_log))
else:
    print("No control log found. Controls will have limited metadata unless later enriched.")

# Scan folders
test_files = list_pdfs(OA_TEST_DIR)
ctrl_files = list_pdfs(OA_CONTROL_DIR)

print("Found TEST PDFs:", len(test_files))
print("Found CONTROL PDFs:", len(ctrl_files))

# Load existing DB (append-only)
if os.path.exists(RAW_DB_CSV):
    db = pd.read_csv(RAW_DB_CSV, dtype=str).fillna("")
    print("Existing RAW_DB loaded:", len(db))
else:
    db = pd.DataFrame()

existing_ids = set(db["paper_id"].tolist()) if (not db.empty and "paper_id" in db.columns) else set()

new_rows = []

# Build TEST rows
for fn in tqdm(test_files, desc="Index TEST PDFs", unit="pdf"):
    code = parse_test_code(fn)
    paper_id = f"test:{code}"
    if paper_id in existing_ids:
        continue

    pdf_path = os.path.join(OA_TEST_DIR, fn)

    if code in rw_by_code.index:
        r = rw_by_code.loc[code]
        row = {
            "paper_id": paper_id,
            "class_label": "test",
            "case_code": code,
            "source_pdf_path": pdf_path,
            "pdf_sha1": sha1_file_safe(pdf_path),

            "title": r.get("Title",""),
            "doi": r.get("DOI",""),
            "doi_norm": r.get("doi_norm",""),
            "journal": r.get("Journal",""),
            "publisher": r.get("Publisher",""),
            "subject": r.get("Subject",""),
            "article_type": r.get("ArticleType",""),
            "year": r.get("year",""),
            "case_country_raw": r.get("Country",""),
            "case_country_iso2": r.get("case_country_iso2",""),
        }
    else:
        row = {
            "paper_id": paper_id,
            "class_label": "test",
            "case_code": code,
            "source_pdf_path": pdf_path,
            "pdf_sha1": sha1_file_safe(pdf_path),

            "title": "",
            "doi": "",
            "doi_norm": "",
            "journal": "",
            "publisher": "",
            "subject": "",
            "article_type": "",
            "year": "",
            "case_country_raw": "",
            "case_country_iso2": "",
        }

    new_rows.append(row)

# Build CONTROL rows
for fn in tqdm(ctrl_files, desc="Index CONTROL PDFs", unit="pdf"):
    case_code = parse_control_case_code(fn)
    paper_id = f"control:{case_code}"
    if paper_id in existing_ids:
        continue

    pdf_path = os.path.join(OA_CONTROL_DIR, fn)

    row = {
        "paper_id": paper_id,
        "class_label": "control",
        "case_code": case_code,
        "source_pdf_path": pdf_path,
        "pdf_sha1": sha1_file_safe(pdf_path),

        "title": "",
        "doi": "",
        "doi_norm": "",
        "journal": "",
        "publisher": "",
        "subject": "",
        "article_type": "",
        "year": "",
        "case_country_raw": "",
        "case_country_iso2": "",
    }

    if control_log is not None:
        hit = control_log[control_log["case_code"].astype(str).str.strip() == str(case_code).strip()]
        hit = hit[hit["status"].str.lower().eq("downloaded")]
        if not hit.empty:
            h0 = hit.iloc[-1]
            row["doi"] = h0.get("control_doi","")
            row["doi_norm"] = norm_doi(row["doi"])
            row["title"] = h0.get("control_title","")
            row["year"] = h0.get("control_year","")
            row["article_type"] = h0.get("control_type","")
            row["journal"] = h0.get("control_journal","")
            row["publisher"] = h0.get("control_publisher","")
            row["subject"] = h0.get("control_subject","")
            row["case_country_iso2"] = h0.get("case_country_iso2","")

    new_rows.append(row)

new_df = pd.DataFrame(new_rows).fillna("")
print("New rows to append:", len(new_df))

# Ensure extraction/output columns exist for new rows
EXTRA_COLS = [
    "section_title","section_abstract","section_introduction","section_methods","section_results","section_discussion",
    "section_map_version","extraction_source","extraction_notes",
    "quality_percent","quality_rationale",
    "applicability","not_applicable_reason",
    "confidence_percent"
]
for c in EXTRA_COLS:
    if c not in new_df.columns:
        new_df[c] = ""

# Append
if db.empty:
    db = new_df.copy()
else:
    db = pd.concat([db, new_df], ignore_index=True)

db.to_csv(RAW_DB_CSV, index=False)
print("RAW_DB updated:", len(db), "rows")
print("Saved to:", RAW_DB_CSV)

In [ ]:
# 8) Section Canonicalisation Rules

# Canonical sections
CANON = ["title","abstract","introduction","methods","results","discussion"]

# Rule-based seed aliases (extend anytime)
RULE_ALIASES = {
    "title": ["title"],
    "abstract": ["abstract","summary","structured abstract"],
    "introduction": ["introduction","background","background and aims","overview"],
    "methods": ["methods","materials and methods","methodology","experimental",
                "patients and methods","study design","statistical analysis"],
    "results": ["results","findings","outcomes","experiments","analysis"],
    "discussion": ["discussion","conclusion","conclusions","discussion and conclusions",
                   "discussion/conclusion","general discussion","interpretation"],
}

def norm_text(x):
    x = str(x or "").strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x

def is_heading_like(line):
    s = line.strip()
    if len(s) < 3 or len(s) > 80:
        return False
    if re.fullmatch(r"[\d\W]+", s):
        return False
    if s.isupper() and len(s) <= 60:
        return True
    if re.match(r"^\d+(\.\d+)*\s+[A-Za-z]", s):
        return True
    words = s.split()
    if 1 <= len(words) <= 8:
        caps = sum(1 for w in words if len(w)>2 and w[0].isupper())
        if caps >= max(1, len(words)//2):
            return True
    return False

def extract_candidate_headings(pdf_path, max_pages=3):
    heads = []
    doc = None
    try:
        doc = fitz.open(pdf_path)
        n = min(len(doc), max_pages)
        for i in range(n):
            text = doc[i].get_text("text")
            for line in text.splitlines():
                if is_heading_like(line):
                    heads.append(line.strip())
    except Exception:
        return []
    finally:
        try:
            if doc is not None:
                doc.close()
        except Exception:
            pass

    out = []
    for h in heads:
        h2 = re.sub(r"^\d+(\.\d+)*\s+", "", h).strip()
        h2 = re.sub(r"[:\.\-–—]+$", "", h2).strip()
        if h2:
            out.append(h2)
    return out

# Robust local caching
LOCAL_HEADINGS_CACHE = "/content/pdf_cache_headings"
os.makedirs(LOCAL_HEADINGS_CACHE, exist_ok=True)

def safe_copy_to_local(src_path):
    key = hashlib.sha1(src_path.encode("utf-8")).hexdigest()
    dst = os.path.join(LOCAL_HEADINGS_CACHE, f"{key}.pdf")
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        return dst
    last = None
    for i in range(4):
        try:
            shutil.copyfile(src_path, dst)
            if os.path.getsize(dst) > 0:
                return dst
        except Exception as e:
            last = e
            time.sleep(1.0 * (i + 1))
    raise last if last else RuntimeError("copy failed")

# Files
CHECKPOINT_FILE = os.path.join(os.path.dirname(HEADING_SCAN_CSV), "heading_scan_checkpoint.txt")

# Load DB
db = pd.read_csv(RAW_DB_CSV, dtype=str).fillna("")
paths = db["source_pdf_path"].dropna().tolist()
paths = [p for p in paths if isinstance(p, str) and p and os.path.exists(p)]

# Load checkpoint (already scanned PDFs)
done_set = set()
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        done_set = set([ln.strip() for ln in f.read().splitlines() if ln.strip()])

new_paths = [p for p in paths if p not in done_set]
print("Total PDFs in DB:", len(paths))
print("Already scanned for headings:", len(done_set))
print("NEW PDFs to scan now:", len(new_paths))

if len(new_paths) == 0:
    print("No new PDFs. Skipping heading scan and keeping existing mapping.")
else:

    # Load existing observed headings counts (incremental)
    seen = {}
    if os.path.exists(HEADING_SCAN_CSV):
        prev = pd.read_csv(HEADING_SCAN_CSV, dtype=str).fillna("")
        for _, r in prev.iterrows():
            seen[r["heading_norm"]] = int(float(r["count"])) if str(r["count"]).strip() else 0
        print("Loaded existing observed headings:", len(seen), "unique headings")
    else:
        print("No observed headings file yet. Creating from scratch (incrementally).")

    # Scan ONLY new PDFs
    save_every = 250
    scanned = 0
    newly_done = []

    for p in tqdm(new_paths, desc="Scanning headings (NEW PDFs only)", unit="pdf"):
        try:
            local_p = safe_copy_to_local(p)
            hs = extract_candidate_headings(local_p, max_pages=MAX_PAGES_SCAN_HEADINGS)
            for h in hs:
                hn = norm_text(h)
                if hn:
                    seen[hn] = seen.get(hn, 0) + 1
        except Exception as e:
            print(f"heading scan failed: {p} :: {type(e).__name__}: {e}")

        done_set.add(p)
        newly_done.append(p)
        scanned += 1

        # periodic save of counts + checkpoint
        if (scanned % save_every) == 0:
            obs = pd.DataFrame(
                [{"heading_norm": k, "count": v} for k, v in sorted(seen.items(), key=lambda x: x[1], reverse=True)]
            )
            obs.to_csv(HEADING_SCAN_CSV, index=False)

            with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f:
                f.write("\n".join(newly_done) + "\n")
            newly_done = []

    # final save
    obs = pd.DataFrame(
        [{"heading_norm": k, "count": v} for k, v in sorted(seen.items(), key=lambda x: x[1], reverse=True)]
    )
    obs.to_csv(HEADING_SCAN_CSV, index=False)

    if newly_done:
        with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f:
            f.write("\n".join(newly_done) + "\n")

    print("Observed headings updated:", HEADING_SCAN_CSV, "unique headings:", len(obs))

    # Update mapping incrementally
    existing_map = pd.DataFrame(columns=["canonical_section","alias_heading_norm","source","map_version"])
    if os.path.exists(HEADING_MAP_CSV):
        existing_map = pd.read_csv(HEADING_MAP_CSV, dtype=str).fillna("")
        existing_aliases = set(existing_map["alias_heading_norm"].tolist())
        print("Existing mapping loaded:", len(existing_map), "rows")
    else:
        existing_aliases = set()

    # Always ensure rule aliases exist
    mapping_rows = []
    for canon, aliases in RULE_ALIASES.items():
        for a in aliases:
            mapping_rows.append({"canonical_section": canon, "alias_heading_norm": norm_text(a), "source": "rule"})

    rule_df = pd.DataFrame(mapping_rows).drop_duplicates()
    # merge with existing
    combined = pd.concat([existing_map, rule_df], ignore_index=True).drop_duplicates(subset=["canonical_section","alias_heading_norm"])

    combined_aliases = set(combined["alias_heading_norm"].tolist())

    def best_canon_for_heading(h_norm):
        best = ("", 0)
        for canon, aliases in RULE_ALIASES.items():
            for a in aliases:
                sc = fuzz.token_set_ratio(h_norm, norm_text(a))
                if sc > best[1]:
                    best = (canon, sc)
        return best

    # Only consider headings not already in mapping
    unmapped = [h for h in obs["heading_norm"].tolist() if h not in combined_aliases]

    auto_added = []
    for h_norm in unmapped:
        canon, sc = best_canon_for_heading(h_norm)
        if sc >= 92:
            auto_added.append({"canonical_section": canon, "alias_heading_norm": h_norm, "source": f"auto_fuzzy_{sc}"})

    auto_df = pd.DataFrame(auto_added).drop_duplicates()
    if auto_df.empty:
        print("No new auto-mappings added (nothing new to map). Keeping mapping file unchanged.")
    else:
        MAP_VERSION = time.strftime("%Y%m%d_%H%M%S")
        auto_df["map_version"] = MAP_VERSION

        combined = pd.concat([combined, auto_df], ignore_index=True).drop_duplicates(subset=["canonical_section","alias_heading_norm"])
        # For older rows missing map_version, fill with last known or "legacy"
        if "map_version" not in combined.columns:
            combined["map_version"] = MAP_VERSION
        combined["map_version"] = combined["map_version"].replace("", MAP_VERSION)

        combined.to_csv(HEADING_MAP_CSV, index=False)
        print(f"Mapping updated with {len(auto_df)} new aliases. Saved:", HEADING_MAP_CSV)

In [ ]:
# 9) Runtime Controls & Performance Settings

# Silence trafilatura noise (403/410 spam)
logging.getLogger("trafilatura").setLevel(logging.CRITICAL)
logging.getLogger("courlan").setLevel(logging.CRITICAL)

# 9.1) Robustness / performance knobs
MAX_SECONDS_PER_PAPER = 90          # hard stop per paper (prevents hangs)
MAX_PAPERS_PER_RUN = None           # process ALL remaining in one run
SAVE_EVERY_N = 50                   # frequent saves for restart safety
LOCAL_CACHE_DIR = "/content/pdf_cache"
LOCAL_OA_DIR = "/content/oa_cache"
GROBID_TIMEOUT = 45                 # seconds (per GROBID call)
GROBID_RETRIES = 2
PDF_COPY_RETRIES = 4
REQUEST_TIMEOUT = 15
MAX_PAGES_EXTRACT_TEXT = None       # set 12 for speed if needed
PDF_CHAR_LIMIT_PER_PAPER = 2_000_000

# Internet fallback controls
USE_INTERNET_FALLBACK = True
TRY_PMC = True                      # best reliability
TRY_UNPAYWALL = True                # good reliability
TRY_OPENALEX_PDF_URL = True         # occasionally helpful
MIN_WEB_TEXT_CHARS = 2000           # accept web text if >= this
MIN_SECTION_TOTAL_CHARS_GOOD = 1500 # accept sectioned content if >= this

os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)
os.makedirs(LOCAL_OA_DIR, exist_ok=True)

# 9.2) Load DB
db = pd.read_csv(RAW_DB_CSV, dtype=str).fillna("")

# Ensure expected columns exist (for old CSVs)
NEEDED_COLS = [
    "paper_id","class_label","case_code","source_pdf_path","title","doi","article_type",
    "section_title","section_abstract","section_introduction","section_methods","section_results","section_discussion",
    "section_map_version","extraction_source","extraction_notes",
    "quality_percent","quality_rationale","confidence_percent",
    "applicability","not_applicable_reason"
]
for c in NEEDED_COLS:
    if c not in db.columns:
        db[c] = ""

# 9.3) Load heading mapping
sec_map = pd.read_csv(HEADING_MAP_CSV, dtype=str).fillna("") if os.path.exists(HEADING_MAP_CSV) else pd.DataFrame()
CURRENT_MAP_VERSION = sec_map["map_version"].iloc[0] if (not sec_map.empty and "map_version" in sec_map.columns) else ""

def norm_text(x):
    x = str(x or "").strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x

def norm_doi(x):
    x = norm_text(x)
    x = x.replace("https://doi.org/", "").replace("http://doi.org/", "")
    return x

alias_to_canon = {}
if not sec_map.empty:
    for _, r in sec_map.iterrows():
        a = r.get("alias_heading_norm","")
        c = r.get("canonical_section","")
        if a and c:
            alias_to_canon[a] = c

CANON = ["title","abstract","introduction","methods","results","discussion"]
CANON_SET = set(CANON)

# 9.4) Heading detection + splitting
def is_heading_like(line):
    s = line.strip()
    if len(s) < 3 or len(s) > 80:
        return False
    if re.fullmatch(r"[\d\W]+", s):
        return False
    if s.isupper() and len(s) <= 60:
        return True
    if re.match(r"^\d+(\.\d+)*\s+[A-Za-z]", s):
        return True
    words = s.split()
    if 1 <= len(words) <= 8:
        caps = sum(1 for w in words if len(w)>2 and w[0].isupper())
        if caps >= max(1, len(words)//2):
            return True
    return False

def pdf_full_text_fast(pdf_path, max_pages=None, char_cap=2_000_000):
    try:
        doc = fitz.open(pdf_path)
        n_pages = len(doc) if max_pages is None else min(len(doc), max_pages)
        parts = []
        for i in range(n_pages):
            t = doc[i].get_text("text")
            # light cleanup (does not recover missing text, just removes obvious overlay tokens)
            t = re.sub(r"(?im)^\s*retracted\s*$", "", t)
            t = re.sub(r"(?im)retracted\s+article", " ", t)
            parts.append(t)
            if sum(len(x) for x in parts) > char_cap:
                break
        doc.close()
        return "\n".join(parts).strip()
    except Exception:
        return ""

def split_by_headings(text):
    if not text:
        return {c:"" for c in CANON}

    sec = {c: [] for c in CANON}
    current = None

    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue

        if is_heading_like(line):
            h = re.sub(r"^\d+(\.\d+)*\s+", "", line).strip()
            h = re.sub(r"[:\.\-–—]+$", "", h).strip()
            hn = norm_text(h)

            if hn in alias_to_canon:
                current = alias_to_canon[hn]
                continue

            # fuzzy fallback (only if we have a mapping)
            if alias_to_canon:
                best = ("", 0)
                for alias, canon in alias_to_canon.items():
                    sc = fuzz.token_set_ratio(hn, alias)
                    if sc > best[1]:
                        best = (canon, sc)
                if best[1] >= 93:
                    current = best[0]
                    continue

        if current in CANON_SET:
            sec[current].append(line)

    if all(len(v)==0 for v in sec.values()):
        sec["introduction"] = [text]

    return {k: "\n".join(v).strip() for k,v in sec.items()}

# 9.5) Localize PDFs (avoid Drive disconnects)
def localize_pdf(src_path, paper_id):
    dst = os.path.join(LOCAL_CACHE_DIR, paper_id.replace(":","_") + ".pdf")
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        return dst
    last = None
    for i in range(PDF_COPY_RETRIES):
        try:
            shutil.copyfile(src_path, dst)
            if os.path.getsize(dst) > 0:
                return dst
        except Exception as e:
            last = e
            time.sleep(1.0 * (i + 1))
    raise last if last else RuntimeError("copy failed")

# 9.6) GROBID health + restart
def grobid_is_alive():
    try:
        r = requests.get("http://localhost:8070/api/isalive", timeout=2)
        return r.status_code == 200
    except Exception:
        return False

def restart_grobid_no_docker():
    if not os.path.exists("/content/grobid"):
        return False
    try:
        subprocess.call("pkill -f 'grobid.*jetty' >/dev/null 2>&1 || true", shell=True)
        subprocess.call("pkill -f 'org.eclipse.jetty' >/dev/null 2>&1 || true", shell=True)
        time.sleep(2)
        subprocess.call("cd /content/grobid && nohup ./gradlew -q run > /content/grobid_server.log 2>&1 &", shell=True)
        for _ in range(30):
            if grobid_is_alive():
                return True
            time.sleep(1)
        return False
    except Exception:
        return False

def grobid_sections_from_pdf(pdf_path):
    tei = grobid_process_fulltext(pdf_path, timeout=GROBID_TIMEOUT)
    secs, note = grobid_extract_sections_from_tei(tei)
    total = sum(len((secs.get(k,"") or "").strip()) for k in ["abstract","introduction","methods","results","discussion"])
    if total >= MIN_SECTION_TOTAL_CHARS_GOOD:
        return secs, "grobid:tei_ok"
    return secs, f"grobid:weak:{total}"

# 9.7) Applicability + quality
def applicability_from_metadata(article_type):
    at = norm_text(article_type)
    if not at:
        return "applicable", ""
    non_research = {"editorial","letter","retraction","correction","erratum","news","commentary"}
    if any(x in at for x in non_research):
        return "not applicable", f"article_type={article_type}"
    return "applicable", ""

def quality_score(sections, source_note=""):
    lens = {k: len((v or "").strip()) for k,v in sections.items()}
    non_empty = sum(1 for k in ["abstract","introduction","methods","results","discussion"] if lens.get(k,0) >= 200)

    score = 10 + non_empty * 18
    used_web = isinstance(source_note, str) and (source_note.startswith("pmc:") or source_note.startswith("unpaywall:") or source_note.startswith("openalex:"))
    used_grobid = isinstance(source_note, str) and source_note.startswith("grobid:")

    if used_web: score += 10
    if used_grobid: score += 15
    for k in ["introduction","methods","results","discussion"]:
        if lens.get(k,0) < 500:
            score -= 10
    score = max(0, min(100, score))

    conf = 55
    heading_detected = any(lens.get(k,0)>0 for k in ["abstract","methods","results","discussion"])
    if heading_detected: conf += 20
    if used_web: conf += 15
    if used_grobid: conf += 20
    if non_empty >= 4: conf += 10
    conf = max(0, min(100, conf))

    rationale = f"non_empty_sections={non_empty}/5; heading_detected={heading_detected}; source={source_note}; lens={lens}"
    return str(score), str(conf), rationale

# 9.8) Internet fallback (PMC + Unpaywall + OpenAlex PDF URL)
OPENALEX = "https://api.openalex.org"
UNPAYWALL = "https://api.unpaywall.org/v2"

def _req_get(url, params=None, timeout=REQUEST_TIMEOUT):
    try:
        r = requests.get(url, params=params, timeout=timeout, headers={"User-Agent":"Mozilla/5.0"})
        return r
    except Exception:
        return None

def openalex_work_by_doi(doi_norm):
    if not doi_norm:
        return None
    url = f"{OPENALEX}/works/https://doi.org/{doi_norm}"
    params = {"mailto": OPENALEX_MAILTO}
    r = _req_get(url, params=params, timeout=12)
    if r is None:
        return None
    if r.status_code == 404:
        return None
    try:
        r.raise_for_status()
        return r.json()
    except Exception:
        return None

def openalex_pdf_urls(work_json):
    urls = []
    if not work_json:
        return urls
    pl = work_json.get("primary_location") or {}
    if pl.get("pdf_url"):
        urls.append(pl["pdf_url"])
    for loc in (work_json.get("locations") or []):
        if loc.get("pdf_url"):
            urls.append(loc["pdf_url"])
    # keep unique, http only
    out = []
    for u in urls:
        if isinstance(u,str) and u.startswith("http") and u not in out:
            out.append(u)
    return out

def pmc_id_from_openalex(work_json):
    if not work_json:
        return ""
    ids = work_json.get("ids") or {}
    pmc = ids.get("pmc") or ""
    if isinstance(pmc, str) and "PMC" in pmc:
        m = re.search(r"(PMC\d+)", pmc)
        if m:
            return m.group(1)
    for loc in (work_json.get("locations") or []):
        u = loc.get("landing_page_url") or ""
        m = re.search(r"(PMC\d+)", u)
        if m:
            return m.group(1)
    return ""

def fetch_pmc_text(pmcid):
    if not pmcid:
        return "", "pmc:no_pmcid"
    url = f"https://pmc.ncbi.nlm.nih.gov/articles/{pmcid}/"
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return "", "pmc:fetch_failed"
        text = trafilatura.extract(downloaded)
        if text and len(text) >= MIN_WEB_TEXT_CHARS:
            return text, f"pmc:html:{pmcid}"
        return "", "pmc:too_short"
    except Exception as e:
        return "", f"pmc:error:{type(e).__name__}"

def fetch_unpaywall_best(doi_norm):
    if not doi_norm:
        return "", "unpaywall:no_doi"
    url = f"{UNPAYWALL}/{doi_norm}"
    params = {"email": OPENALEX_MAILTO}
    r = _req_get(url, params=params, timeout=12)
    if r is None:
        return "", "unpaywall:request_failed"
    if r.status_code == 404:
        return "", "unpaywall:not_found"
    try:
        r.raise_for_status()
        j = r.json()
    except Exception:
        return "", f"unpaywall:bad_response:{r.status_code}"

    best = j.get("best_oa_location") or {}
    oa_url = best.get("url_for_pdf") or best.get("url") or ""
    if not oa_url:
        return "", "unpaywall:no_oa_url"

    return oa_url, "unpaywall:oa_url"

def download_pdf(url, out_path, timeout=20, max_bytes=40_000_000):
    """
    Best-effort PDF download with sanity checks.
    """
    try:
        r = requests.get(url, timeout=timeout, headers={"User-Agent":"Mozilla/5.0"}, stream=True)
        if r.status_code != 200:
            return False
        total = 0
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*64):
                if not chunk:
                    continue
                f.write(chunk)
                total += len(chunk)
                if total > max_bytes:
                    return False
        # basic PDF signature check
        with open(out_path, "rb") as f:
            sig = f.read(4)
        return sig == b"%PDF"
    except Exception:
        return False

def try_best_internet_fallback(doi_norm, paper_id, deadline_ts):
    """
    Returns (mode, payload, note):
      mode="text": payload=text
      mode="pdf":  payload=local_pdf_path
      mode="none": payload=""
    """
    if not doi_norm:
        return "none", "", "internet:no_doi"
    if time.time() > deadline_ts:
        return "none", "", "internet:deadline"

    w = openalex_work_by_doi(doi_norm)

    # 1) PMC (very reliable when present)
    if TRY_PMC and time.time() < deadline_ts:
        pmcid = pmc_id_from_openalex(w)
        if pmcid:
            text, note = fetch_pmc_text(pmcid)
            if text:
                return "text", text, note

    # 2) Unpaywall best OA (often gives direct OA URL)
    if TRY_UNPAYWALL and time.time() < deadline_ts:
        oa_url, note = fetch_unpaywall_best(doi_norm)
        if oa_url:
            # If PDF link, download locally and run GROBID/PDF on it
            if oa_url.lower().endswith(".pdf"):
                local_pdf = os.path.join(LOCAL_OA_DIR, f"{paper_id.replace(':','_')}_oa.pdf")
                ok = download_pdf(oa_url, local_pdf, timeout=20)
                if ok:
                    return "pdf", local_pdf, note
                # If PDF download fails, don't spam; move on
            else:
                # HTML: attempt trafilatura extraction (many OA pages work)
                try:
                    downloaded = trafilatura.fetch_url(oa_url)
                    if downloaded:
                        text = trafilatura.extract(downloaded)
                        if text and len(text) >= MIN_WEB_TEXT_CHARS:
                            return "text", text, "unpaywall:html_text"
                except Exception:
                    pass

    # 3) OpenAlex pdf_url (sometimes available)
    if TRY_OPENALEX_PDF_URL and time.time() < deadline_ts:
        for u in openalex_pdf_urls(w)[:2]:
            local_pdf = os.path.join(LOCAL_OA_DIR, f"{paper_id.replace(':','_')}_oa2.pdf")
            ok = download_pdf(u, local_pdf, timeout=20)
            if ok:
                return "pdf", local_pdf, "openalex:pdf_url"

    return "none", "", "internet:no_oa_source"

# 9.9) Decide which rows need extraction
needs = db[
    (db["section_abstract"].eq("")) &
    (db["section_introduction"].eq("")) &
    (db["section_methods"].eq("")) &
    (db["section_results"].eq("")) &
    (db["section_discussion"].eq(""))
].copy()

print("Rows needing extraction:", len(needs))

to_process = needs.index.tolist()
if MAX_PAPERS_PER_RUN is not None:
    to_process = to_process[:MAX_PAPERS_PER_RUN]
print(f"Will process this run: {len(to_process)} / {len(needs)} needing extraction")

# Ensure GROBID is up (best effort)
if not grobid_is_alive():
    restart_grobid_no_docker()

# 9.10) Main extraction loop
for n, idx in enumerate(tqdm(to_process, desc="Extract sections (robust)", unit="paper"), start=1):
    t0 = time.time()
    deadline_ts = t0 + MAX_SECONDS_PER_PAPER

    row = db.loc[idx]
    paper_id = row.get("paper_id","")
    src_pdf = row.get("source_pdf_path","")
    doi_norm = norm_doi(row.get("doi",""))

    # defaults
    db.at[idx, "section_map_version"] = CURRENT_MAP_VERSION

    # applicability
    appl, reason = applicability_from_metadata(row.get("article_type",""))
    db.at[idx, "applicability"] = appl
    db.at[idx, "not_applicable_reason"] = reason

    try:
        if not src_pdf or not os.path.exists(src_pdf):
            db.at[idx, "extraction_source"] = "missing_pdf"
            db.at[idx, "extraction_notes"] = "pdf_not_found"
            db.at[idx, "quality_percent"] = "0"
            db.at[idx, "confidence_percent"] = "0"
            db.at[idx, "quality_rationale"] = "PDF missing."
            continue

        # 1) Localize PDF (Drive robustness)
        if time.time() > deadline_ts:
            raise TimeoutError("deadline before localization")
        local_pdf = localize_pdf(src_pdf, paper_id)

        secs = None
        source_note = ""

        # 2) GROBID first (best sectioning)
        for attempt in range(GROBID_RETRIES + 1):
            if time.time() > deadline_ts:
                raise TimeoutError("deadline during GROBID")
            if not grobid_is_alive():
                restart_grobid_no_docker()
            try:
                g_secs, g_note = grobid_sections_from_pdf(local_pdf)
                secs = g_secs
                source_note = g_note
                if g_note.startswith("grobid:tei_ok"):
                    break
            except Exception as e:
                source_note = f"grobid:error:{type(e).__name__}"
                time.sleep(1)

        # 3) If GROBID weak, try PDF text layer (fast, local)
        def total_chars(sdict):
            return sum(len((sdict.get(k,"") or "").strip()) for k in ["abstract","introduction","methods","results","discussion"])

        if (secs is None) or (not source_note.startswith("grobid:tei_ok") and total_chars(secs) < 800):
            if time.time() > deadline_ts:
                raise TimeoutError("deadline before PDF fallback")
            text = pdf_full_text_fast(local_pdf, max_pages=MAX_PAGES_EXTRACT_TEXT, char_cap=PDF_CHAR_LIMIT_PER_PAPER)
            if text and len(text) >= 500:
                secs2 = split_by_headings(text)
                # accept if better than current
                if secs is None or total_chars(secs2) > total_chars(secs):
                    secs = secs2
                    source_note = "pdf:pymupdf"

        # 4) If still poor, use robust Internet fallback (PMC/Unpaywall/OpenAlex PDF URL)
        if USE_INTERNET_FALLBACK and (secs is None or total_chars(secs) < 800) and (time.time() < deadline_ts - 10):
            mode, payload, note = try_best_internet_fallback(doi_norm, paper_id, deadline_ts)
            if mode == "text" and payload:
                secs_web = split_by_headings(payload)
                if secs is None or total_chars(secs_web) > total_chars(secs):
                    secs = secs_web
                    source_note = note
            elif mode == "pdf" and payload and os.path.exists(payload):
                # Run GROBID on OA PDF (best)
                if time.time() < deadline_ts - 5:
                    try:
                        g_secs, g_note = grobid_sections_from_pdf(payload)
                        if secs is None or total_chars(g_secs) > total_chars(secs):
                            secs = g_secs
                            source_note = f"{note}|{g_note}"
                    except Exception:
                        pass
                # If still weak, try PyMuPDF on OA PDF
                if secs is None or total_chars(secs) < 800:
                    t = pdf_full_text_fast(payload, max_pages=MAX_PAGES_EXTRACT_TEXT, char_cap=PDF_CHAR_LIMIT_PER_PAPER)
                    if t and len(t) >= 500:
                        secs_webpdf = split_by_headings(t)
                        if secs is None or total_chars(secs_webpdf) > total_chars(secs):
                            secs = secs_webpdf
                            source_note = f"{note}|pdf:pymupdf(oa)"

        # 5) If still nothing usable, mark and continue
        if secs is None or total_chars(secs) < 200:
            db.at[idx, "extraction_source"] = source_note or "no_text"
            db.at[idx, "extraction_notes"] = "no_text_or_too_short"
            db.at[idx, "quality_percent"] = "0"
            db.at[idx, "confidence_percent"] = "10"
            db.at[idx, "quality_rationale"] = "No usable text from GROBID/PDF/PMC/Unpaywall."
            continue

        # 6) Write sections
        db.at[idx, "section_title"] = row.get("title","")
        db.at[idx, "section_abstract"] = secs.get("abstract","")
        db.at[idx, "section_introduction"] = secs.get("introduction","")
        db.at[idx, "section_methods"] = secs.get("methods","")
        db.at[idx, "section_results"] = secs.get("results","")
        db.at[idx, "section_discussion"] = secs.get("discussion","")

        db.at[idx, "extraction_source"] = source_note
        db.at[idx, "extraction_notes"] = "ok"

        q, conf, why = quality_score(secs, source_note=source_note)
        db.at[idx, "quality_percent"] = q
        db.at[idx, "confidence_percent"] = conf
        db.at[idx, "quality_rationale"] = why

        # Mark not applicable if no IMRAD sections detected
        if db.at[idx, "applicability"] == "applicable":
            nonempty = sum(1 for k in ["abstract","methods","results","discussion"]
                           if len((secs.get(k,"") or "").strip()) >= 200)
            if nonempty == 0:
                db.at[idx, "applicability"] = "not applicable"
                db.at[idx, "not_applicable_reason"] = "no_imrad_sections_detected"

    except TimeoutError:
        db.at[idx, "extraction_source"] = db.at[idx, "extraction_source"] or "timeout"
        db.at[idx, "extraction_notes"] = "timeout"
        db.at[idx, "quality_percent"] = "0"
        db.at[idx, "confidence_percent"] = "0"
        db.at[idx, "quality_rationale"] = f"Per-paper time limit exceeded ({MAX_SECONDS_PER_PAPER}s)."
    except Exception as e:
        db.at[idx, "extraction_source"] = f"error:{type(e).__name__}"
        db.at[idx, "extraction_notes"] = "error"
        db.at[idx, "quality_percent"] = "0"
        db.at[idx, "confidence_percent"] = "0"
        db.at[idx, "quality_rationale"] = f"Exception: {type(e).__name__}: {e}"

    # frequent save
    if (n % SAVE_EVERY_N) == 0:
        db.to_csv(RAW_DB_CSV, index=False)

# final save
db.to_csv(RAW_DB_CSV, index=False)
print("Robust extraction pass complete. Saved:", RAW_DB_CSV)
print("Local cache used:", LOCAL_CACHE_DIR)
print("OA cache used:", LOCAL_OA_DIR)

In [ ]:
# 10) Add/refresh control_match_level in RAW_DB

BASE_DIR = "/content/drive/MyDrive/Dissertation"
RAW_DB_CSV = os.path.join(BASE_DIR, "raw_fulltext_db.csv")
CONTROL_LOG_CSV = os.path.join(BASE_DIR, "control_download_log_all_new.csv")

db = pd.read_csv(RAW_DB_CSV, dtype=str).fillna("")

# Ensure the new column exists
if "control_match_level" not in db.columns:
    db["control_match_level"] = ""

# Load control log (contains match_level)
if not os.path.exists(CONTROL_LOG_CSV):
    raise FileNotFoundError(f"Control log not found: {CONTROL_LOG_CSV}")

cl = pd.read_csv(CONTROL_LOG_CSV, dtype=str).fillna("")
# Keep only downloaded controls, and keep the latest record per case_code (if duplicates exist)
cl["case_code"] = cl["case_code"].astype(str).str.strip()
cl["status_norm"] = cl["status"].astype(str).str.strip().str.lower()
cl_ok = cl[cl["status_norm"].eq("downloaded")].copy()

# Latest per case_code (works even if the log has multiple rows per case)
cl_ok = cl_ok.drop_duplicates(subset=["case_code"], keep="last")

# Build lookup: case_code -> match_level
match_lookup = dict(zip(cl_ok["case_code"], cl_ok.get("match_level", "")))

# Apply only to controls (paper_id starts with "control:" or class_label == "control")
is_control = (db["class_label"].astype(str).str.lower().eq("control")) | (db["paper_id"].astype(str).str.startswith("control:"))
db.loc[is_control, "control_match_level"] = db.loc[is_control, "case_code"].astype(str).str.strip().map(match_lookup).fillna("")

# Quick audit
n_controls = int(is_control.sum())
n_filled = int((db.loc[is_control, "control_match_level"].astype(str).str.strip() != "").sum())
print(f"Controls in DB: {n_controls}")
print(f"Controls w/ match level filled: {n_filled}")
print(f"Controls still missing match level: {n_controls - n_filled}")

# Save back
db.to_csv(RAW_DB_CSV, index=False)
print("Saved RAW_DB with control_match_level:", RAW_DB_CSV)

In [ ]:
# 11) Quick audit tables

db = pd.read_csv(RAW_DB_CSV, dtype=str).fillna("")

# level-of-quality buckets
db["quality_num"] = pd.to_numeric(db["quality_percent"], errors="coerce").fillna(0)

summary = db.groupby("class_label").agg(
    n=("paper_id","count"),
    applicable=("applicability", lambda s: (s=="applicable").sum()),
    not_applicable=("applicability", lambda s: (s=="not applicable").sum()),
    mean_quality=("quality_num","mean"),
    median_quality=("quality_num","median"),
).reset_index()

quality_bins = pd.cut(db["quality_num"], bins=[-1,0,25,50,75,100], labels=["0","1-25","26-50","51-75","76-100"])
by_label_bins = pd.crosstab(db["class_label"], quality_bins)

missing_sections = {}
for sec in ["abstract","introduction","methods","results","discussion"]:
    missing_sections[sec] = (db[f"section_{sec}"].str.strip()=="").sum()

missing_table = pd.DataFrame([missing_sections]).T.reset_index()
missing_table.columns = ["section","missing_count"]

print("=== Summary by class ===")
display(summary)

print("\n=== Quality distribution by class ===")
display(by_label_bins)

print("\n=== Missing sections (overall) ===")
display(missing_table)

print("\n=== Mapping version used ===")
display(db["section_map_version"].value_counts().head(10))